# jA_02 - FL near-domain di 3 EC2 (JURNAL, Konfig A)

Menjalankan **federated learning near-domain** (Client-CIC2017 + Client-CIC2018,
~76 fitur) di 3 instance EC2, **dari dalam notebook SageMaker ini**. Notebook hanya
memanggil orchestrator `federated/aws/run_journal_neardomain.py` (boto3 + SSM);
Anda tidak perlu membuka terminal.

**Alur yang dilakukan orchestrator otomatis:** upload skrip node -> deploy stack 3 EC2
-> setup node -> jalankan FL tiap seed -> aggregator meng-evaluasi (MCC federated /
centralized / local-only, protokol z-score bebas-kebocoran) -> tulis `eval_results.json`
ke S3 -> (Anda) teardown.

**Prasyarat:** `jA_01_partition_neardomain.ipynb` sudah dijalankan (data ada di
`s3://.../federated_journal/neardomain/data/{cic2017,cic2018}/`).

**Output di S3:** `federated_journal/neardomain/run_J_NEAR_s<seed>/eval_results.json`.
Kiro menariknya ke lokal untuk mengisi tabel jurnal.

> Jalankan di SageMaker. Role instance memakai IAM dgn akses CloudFormation/SSM/S3
> (sama seperti saat run survei ISRITI). JANGAN mengarang angka.

## URUTAN PAKAI (ringkas)
1. Sel 1: pindah ke folder `federated/aws` + cek prasyarat data.
2. Sel 2: **UJI CEPAT** 10 ronde, 1 seed (pastikan pipeline nyambung). ~10-20 menit.
3. Sel 3: **RUN PENUH** 100 ronde, 5 seed (kalau uji cepat sukses). Agak lama.
4. Sel 4: cek status / tail log bila perlu.
5. Sel 5: **TEARDOWN (WAJIB)** setelah semua `eval_results.json` ada.

## Sel 1 - Lokasi skrip + cek prasyarat data di S3

In [ ]:
import os, subprocess, sys

# Cari folder federated/aws relatif terhadap lokasi notebook di SageMaker.
CANDS = ['../../aws', '../../../federated/aws', 'federated/aws', '../aws']
AWS_DIR = next((p for p in CANDS if os.path.exists(os.path.join(p,'run_journal_neardomain.py'))), None)
assert AWS_DIR, ('Tak menemukan federated/aws/run_journal_neardomain.py. '
                 'Pastikan sudah git pull terbaru di SageMaker.')
AWS_DIR = os.path.abspath(AWS_DIR)
print('orchestrator di:', AWS_DIR)

# Pastikan boto3 ada (SageMaker biasanya sudah).
try:
    import boto3
except ImportError:
    subprocess.run([sys.executable,'-m','pip','install','-q','boto3'], check=True)
    import boto3

# Cek data near-domain hasil jA_01 ada di S3.
BUCKET='ssh-detection-features-232032302717'; REGION='ap-southeast-1'
s3=boto3.client('s3',region_name=REGION)
need=['federated_journal/neardomain/data/cic2017/cic2017_train.npz',
      'federated_journal/neardomain/data/cic2017/cic2017_test.npz',
      'federated_journal/neardomain/data/cic2018/cic2018_train.npz',
      'federated_journal/neardomain/data/cic2018/cic2018_test.npz']
missing=[]
for k in need:
    try: s3.head_object(Bucket=BUCKET,Key=k)
    except Exception: missing.append(k)
assert not missing, 'PRASYARAT GAGAL - data jA_01 belum ada di S3: %s'%missing
print('OK: 4 file partisi near-domain siap di S3.')

def orch(args):
    """Panggil orchestrator di folder federated/aws; stream output ke notebook."""
    cmd=[sys.executable,'run_journal_neardomain.py']+args
    print('>>', ' '.join(cmd))
    p=subprocess.run(cmd, cwd=AWS_DIR, capture_output=True, text=True)
    print(p.stdout)
    if p.stderr.strip(): print('[stderr]\n'+p.stderr)
    return p.returncode
print('helper orch() siap')

## Sel 2 - UJI CEPAT (10 ronde, 1 seed)

Deploy stack + setup node + 1 run pendek untuk memastikan semuanya nyambung.
Kalau sukses, akan muncul `run_J_NEAR_s42/eval_results.json` di S3.

In [ ]:
rc = orch(['--rounds','10','--up','--deploy','--setup','--run','--seeds','42'])
print('return code:', rc)

## Sel 3 - RUN PENUH (100 ronde, 5 seed)

Jalankan HANYA bila Sel 2 sukses. Stack sudah ada, jadi tak perlu --deploy/--setup lagi.
Seed 42 (dari uji cepat) otomatis dilewati kalau 100-ronde-nya belum ada? TIDAK:
uji cepat tadi 10 ronde dengan tag yang sama (J_NEAR_s42). Agar run penuh seed 42
tidak terlewati oleh hasil 10-ronde, kita pakai seed 42 juga di 100 ronde dengan
menghapus dulu eval 10-ronde-nya (sel ini melakukannya). Seed lain langsung jalan.

In [ ]:
# Hapus eval 10-ronde seed 42 (uji cepat) agar digantikan hasil 100 ronde.
import boto3
s3=boto3.client('s3',region_name='ap-southeast-1')
for k in ['federated_journal/neardomain/run_J_NEAR_s42/eval_results.json',
          'federated_journal/neardomain/run_J_NEAR_s42/global_final.npz']:
    try: s3.delete_object(Bucket='ssh-detection-features-232032302717',Key=k); print('hapus',k)
    except Exception as e: print('(lewati',k,e,')')

rc = orch(['--rounds','100','--run','--seeds','13','42','101','202','303'])
print('return code:', rc)

## Sel 4 - Status / tail log (opsional, saat menunggu)

In [ ]:
orch(['--status'])
# Contoh lihat log aggregator seed 13 bila sebuah run terasa lama:
# orch(['--tail','agg','--tail-seed','13'])

## Sel 5 - TEARDOWN (WAJIB setelah semua eval_results.json ada)

Menghapus stack 3 EC2 -> biaya nol. Hasil tetap aman di S3.

In [ ]:
# Jalankan HANYA bila Sel 4 menunjukkan kelima seed 'ADA'.
rc = orch(['--teardown'])
print('return code:', rc)